# Read-only multi-facility discovery through RIG

This notebook uses the **experimental** RIG helper in `amsc-client==0.7.1`.

```text
AMSC_TOKEN (AmSC Keycard/PAT) -> RIG -> facility IRI API
```

This is distinct from direct `client.facility(...)`, which uses facility-native authentication. RIG may still require an active facility allocation and configured Credential Vault or token-exchange access. Never paste, print, decode, compare, serialize, or commit a token.

**Validation status (2026-09-28):** this notebook is **statically validated** against the published wheel. The protected reads and optional workflow are **not live-validated** by this repository campaign. Submission is disabled by default.

This notebook performs independent protected probes. It contains no submission, catalog mutation, or filesystem mutation.

In [ ]:
import os
from importlib.metadata import version
from amsc_client import Client
from amsc_client.core.exceptions import ApiError, AuthenticationError

STAGING_API = "https://api.staging.american-science-cloud.org/api/current"
STAGING_RIG = "https://rig.staging.american-science-cloud.org"
AMSC_TOKEN = os.environ["AMSC_TOKEN"]
assert version("amsc-client") == "0.7.1", "Install requirements.txt and restart this kernel"
client = Client(token=AMSC_TOKEN, base_url=STAGING_API)
print("Client release:", version("amsc-client"))

## Runtime inventory
The RIG response, not a hard-coded list, determines which facilities are shown.

In [ ]:
facilities = client.rig_facilities(rig_url=STAGING_RIG)
for meta in facilities:
    print(meta.name, meta.display_name, meta.tier, meta.api_version)

## Independent protected probes
Each facility is isolated so one authentication or API failure does not stop the rest. Messages are bounded categories; arbitrary upstream exception text is not rendered.

In [ ]:
results = {}
for meta in facilities:
    try:
        routed = client.facility_via_rig(meta.name, rig_url=STAGING_RIG)
        visible_projects = routed.projects()
        status = "OK" if visible_projects else "NO PROJECTS"
        results[meta.name] = {"status": status, "client": routed}
    except AuthenticationError:
        results[meta.name] = {"status": "AUTH FAILED", "client": None}
    except ApiError:
        results[meta.name] = {"status": "API FAILED", "client": None}
    except Exception:
        # Keep probing other facilities without rendering arbitrary upstream details.
        results[meta.name] = {"status": "ERROR", "client": None}

for name, result in results.items():
    print(name, result["status"])

## Select a successful facility for read-only inspection
Set `AMSC_FACILITY` to one of the successful names, or leave it unset to choose the first. Only facility metadata and resources are read.

In [ ]:
successful_names = [name for name, result in results.items() if result["status"] in {"OK", "NO PROJECTS"}]
requested = os.getenv("AMSC_FACILITY", "")
selected_name = requested or (successful_names[0] if successful_names else "")
if selected_name not in successful_names:
    raise RuntimeError("No successful routed facility matches AMSC_FACILITY")
selected = results[selected_name]["client"]
print(selected.info())
for resource in selected.resources():
    print(resource.name, resource.resource_type, resource.status)